# 05 — Comparativa automática de modelos

Entrena y evalúa múltiples modelos con los mismos hiperparámetros y compara resultados.
Diseñado para ejecutarse en **Kaggle GPU** sin supervisión.

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers==4.48.0', 'sentencepiece', 'accelerate'], check=True)

import os, gc, json, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report
warnings.filterwarnings('ignore')

# ── Rutas ──────────────────────────────────────────────────────────────────
IN_KAGGLE = os.path.exists('/kaggle/input')
if IN_KAGGLE:
    BASE      = '/kaggle/input/datasets/davidreyesales/hisemotions-2026'
    OUT_BASE  = '/kaggle/working/models'
else:
    BASE      = '../train'
    OUT_BASE  = '../models'

TRAIN_PATH = f'{BASE}/train_augmented.csv' if IN_KAGGLE else '../train/train_augmented.csv'
DEV_PATH   = f'{BASE}/dev.csv'             if IN_KAGGLE else '../dev/dev.csv'
TEST_PATH  = f'{BASE}/test.csv'            if IN_KAGGLE else '../test/test.csv'
os.makedirs(OUT_BASE, exist_ok=True)

# ── Device ─────────────────────────────────────────────────────────────────
if torch.cuda.is_available():              DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():    DEVICE = torch.device('mps')
else:                                      DEVICE = torch.device('cpu')
print(f'Device: {DEVICE} | Kaggle: {IN_KAGGLE}')


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: /Users/davidreyes/Documents/Proyectos/HISEMOTIONS_2026/.venv/bin/python -m pip install --upgrade pip


Device: mps | Kaggle: False


In [2]:
# ── Modelos a entrenar ─────────────────────────────────────────────────────
MODELS = {
    'BETO':     'dccuchile/bert-base-spanish-wwm-cased',
    'mDeBERTa': 'microsoft/mdeberta-v3-base',
    'MarIA':    'IsGarrido/roberta-base-bne',
}

# ── Hiperparámetros ────────────────────────────────────────────────────────
CFG = {
    'max_length':    256,
    'batch_size':    8,     # 16 en Kaggle GPU, 8 para M4 local
    'num_epochs':    10,
    'learning_rate': 2e-5,
    'dropout':       0.3,
    'warmup_ratio':  0.1,
    'weight_decay':  0.01,
    'patience':      3,
    'seed':          42,
    'loss':          'bce',
    'focal_gamma':   2.0,
}

EMOTION_COLS = ['anger', 'fear', 'joy', 'sadness', 'surprise', 'hope']
torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
print(f'Modelos a entrenar: {list(MODELS.keys())}')
print(f'Loss: {CFG["loss"]} | Batch size: {CFG["batch_size"]}')

Modelos a entrenar: ['BETO', 'mDeBERTa', 'MarIA']
Loss: bce | Batch size: 8


In [3]:
def load_split(path):
    df = pd.read_csv(path).dropna(subset=['text']).reset_index(drop=True)
    for col in EMOTION_COLS:
        if col in df.columns:
            df[col] = df[col].fillna(0).astype(int)
    return df

class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.texts  = df['text'].tolist()
        self.labels = (df[EMOTION_COLS].values.astype('float32')
                       if all(c in df.columns for c in EMOTION_COLS) else None)
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_length,
                             padding='max_length', truncation=True, return_tensors='pt')
        item = {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0)}
        if 'token_type_ids' in enc:
            item['token_type_ids'] = enc['token_type_ids'].squeeze(0)
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

class MultiLabelEmotionClassifier(nn.Module):
    def __init__(self, model_name, num_labels=6, dropout=0.3):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        hidden          = self.encoder.config.hidden_size
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_labels)
        # DistilBERT y algunos modelos no aceptan token_type_ids
        self.use_token_type_ids = getattr(
            self.encoder.config, 'type_vocab_size', 0) > 1

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask}
        if token_type_ids is not None and self.use_token_type_ids:
            kwargs['token_type_ids'] = token_type_ids
        out = self.encoder(**kwargs)
        cls = self.dropout(out.last_hidden_state[:, 0, :])
        return self.classifier(cls)

train_df = load_split(TRAIN_PATH)
dev_df   = load_split(DEV_PATH)
test_df  = load_split(TEST_PATH)

counts     = train_df[EMOTION_COLS].sum()
n          = len(train_df)
pos_weight = torch.tensor(
    ((n - counts) / counts.clip(lower=1)).values, dtype=torch.float32).to(DEVICE)

print(f'Train: {len(train_df)} | Dev: {len(dev_df)} | Test: {len(test_df)}')

Train: 2847 | Dev: 425 | Test: 863


In [4]:
import torch.nn.functional as F

class FocalLoss(nn.Module):
    """Binary Focal Loss para multi-label. Penaliza más los ejemplos difíciles."""
    def __init__(self, gamma: float = 2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction='none')
        pt = torch.exp(-bce)
        return ((1 - pt) ** self.gamma * bce).mean()


def build_criterion(cfg, pos_weight):
    if cfg.get('loss', 'bce') == 'focal':
        return FocalLoss(gamma=cfg.get('focal_gamma', 2.0), pos_weight=pos_weight)
    return nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print('FocalLoss y BCEWithLogitsLoss disponibles.')

FocalLoss y BCEWithLogitsLoss disponibles.


In [5]:
# ── Función de entrenamiento completo para un modelo ──────────────────────
def train_model(name, model_name):
    print(f'\n{"="*60}')
    print(f'  {name} — {model_name}')
    print(f'{"="*60}')

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
    except Exception as e:
        print(f'  ERROR cargando tokenizador: {e}')
        return None

    train_loader = DataLoader(EmotionDataset(train_df, tokenizer, CFG['max_length']),
                              batch_size=CFG['batch_size'], shuffle=True,  num_workers=0)
    dev_loader   = DataLoader(EmotionDataset(dev_df,   tokenizer, CFG['max_length']),
                              batch_size=CFG['batch_size'], shuffle=False, num_workers=0)
    test_loader  = DataLoader(EmotionDataset(test_df,  tokenizer, CFG['max_length']),
                              batch_size=CFG['batch_size'], shuffle=False, num_workers=0)

    try:
        model = MultiLabelEmotionClassifier(model_name, dropout=CFG['dropout']).to(DEVICE)
    except Exception as e:
        print(f'  ERROR cargando modelo: {e}')
        return None

    criterion   = build_criterion(CFG, pos_weight)
    total_steps = len(train_loader) * CFG['num_epochs']
    optimizer   = torch.optim.AdamW(model.parameters(), lr=CFG['learning_rate'],
                                    weight_decay=CFG['weight_decay'])
    scheduler   = get_linear_schedule_with_warmup(
        optimizer, int(total_steps * CFG['warmup_ratio']), total_steps)

    def run_epoch(loader, train=True):
        model.train(train)
        total_loss, all_logits, all_labels = 0, [], []
        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for batch in loader:
                ids  = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                tt   = batch.get('token_type_ids')
                if tt is not None: tt = tt.to(DEVICE)
                labs = batch['labels'].to(DEVICE)
                logits = model(ids, mask, tt)
                loss   = criterion(logits, labs)
                if train:
                    optimizer.zero_grad(); loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step(); scheduler.step()
                total_loss += loss.item()
                all_logits.append(logits.detach().cpu().numpy())
                all_labels.append(labs.cpu().numpy())
        return total_loss / len(loader), np.vstack(all_logits), np.vstack(all_labels)

    best_f1, best_thresholds, no_improve = 0.0, np.full(6, 0.5), 0
    model_path = os.path.join(OUT_BASE, f'{name}_best.pt')

    for epoch in range(1, CFG['num_epochs'] + 1):
        train_loss, _, _          = run_epoch(train_loader, train=True)
        _, dev_logits, dev_labels = run_epoch(dev_loader,   train=False)

        probs = 1 / (1 + np.exp(-dev_logits))
        thresholds = np.full(6, 0.5)
        for i in range(6):
            best_t, best_s = 0.5, 0.0
            for t in np.arange(0.10, 0.91, 0.05):
                th = thresholds.copy(); th[i] = t
                s  = f1_score(dev_labels, (probs >= th).astype(int),
                              average='micro', zero_division=0)
                if s > best_s: best_s, best_t = s, t
            thresholds[i] = best_t

        dev_f1 = f1_score(dev_labels, (probs >= thresholds).astype(int),
                          average='micro', zero_division=0)
        print(f'  Epoch {epoch:2d} | loss={train_loss:.4f} | dev F1={dev_f1:.4f}', end='')

        if dev_f1 > best_f1:
            best_f1, best_thresholds, no_improve = dev_f1, thresholds.copy(), 0
            torch.save({'model': model.state_dict(), 'thresholds': thresholds}, model_path)
            print(f'  ✓ guardado')
        else:
            no_improve += 1
            print(f'  ({no_improve}/{CFG["patience"]})')
            if no_improve >= CFG['patience']:
                print(f'  Early stopping en epoch {epoch}.')
                break

    # Evaluar en test con el mejor modelo
    ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model'])
    best_thresholds = ckpt['thresholds']

    _, test_logits, test_labels = run_epoch(test_loader, train=False)
    test_probs = 1 / (1 + np.exp(-test_logits))
    test_preds = (test_probs >= best_thresholds).astype(int)
    test_f1    = f1_score(test_labels, test_preds, average='micro', zero_division=0)

    print(f'\n  Dev F1: {best_f1:.4f} | Test F1: {test_f1:.4f}')

    del model, optimizer, scheduler
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    return {
        'model':      name,
        'hf_name':    model_name,
        'loss':       CFG['loss'],
        'dev_f1':     round(best_f1, 4),
        'test_f1':    round(test_f1, 4),
        'thresholds': best_thresholds.tolist(),
    }

In [ ]:
# ── Ejecutar comparativa ───────────────────────────────────────────────────
results = []

for name, model_name in MODELS.items():
    try:
        result = train_model(name, model_name)
        if result:
            results.append(result)
            pd.DataFrame(results).to_csv(f'{OUT_BASE}/comparison_results.csv', index=False)
    except Exception as e:
        print(f'\n  ERROR en {name}: {e}')
        print(f'  Saltando {name} y continuando...\n')

print('\n✓ Comparativa completada.')


  BETO — dccuchile/bert-base-spanish-wwm-cased


Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Epoch  1 | loss=1.1731 | dev F1=0.4473  ✓ guardado


In [ ]:
# ── Tabla de resultados ────────────────────────────────────────────────────
results_df = pd.DataFrame(results)[['model', 'hf_name', 'dev_f1', 'test_f1']]
results_df = results_df.sort_values('test_f1', ascending=False).reset_index(drop=True)

print('\n=== RESULTADOS FINALES ===')
print(results_df.to_string(index=False))

print(f'\nMejor modelo en test: {results_df.iloc[0]["model"]} '
      f'(Test F1={results_df.iloc[0]["test_f1"]:.4f})')

In [ ]:
import matplotlib.pyplot as plt

x      = np.arange(len(results_df))
width  = 0.35
fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(x - width/2, results_df['dev_f1'],  width, label='Dev F1',  color='steelblue')
ax.bar(x + width/2, results_df['test_f1'], width, label='Test F1', color='coral')

ax.set_xticks(x)
ax.set_xticklabels(results_df['model'], rotation=15)
ax.set_ylabel('Micro F1')
ax.set_title('Comparativa de modelos — HISEMOTIONS 2026')
ax.legend()
ax.set_ylim(0, 0.7)
for bar in ax.patches:
    ax.annotate(f'{bar.get_height():.3f}',
                (bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(f'{OUT_BASE}/comparison_plot.png', dpi=150)
plt.show()